# CMA with MongoDB Atlas

How to give an agent on [Claude Managed Agents](README.md) (CMA) a real database: for
retrieval, for graph traversal, and as its system of record. It runs on MongoDB Atlas with
no platform-level MongoDB integration, using only the standard CMA patterns (custom tools
and MCP toolsets).

Most agent stacks bolt together three or four systems: a vector database for semantic
search, a separate search engine for keywords, a graph store for relationships, and an
operational database for the records themselves. Every seam is another integration, another
credential, another place for the agent's view of the world to drift. MongoDB collapses that
into one engine. The same documents are searchable by **vector** (`$vectorSearch`), by
**full-text** (`$search`), and as a **hybrid** of the two fused with reciprocal rank fusion
(`$rankFusion`), and they are traversable as a **graph** (`$graphLookup`). They are the same
documents the agent reads, writes, and persists decisions to. That is the single-engine
property: one query language, one cluster, one connection. For an agent, it means every
recommendation is grounded in the same data it acts on.

MongoDB plays four roles for a CMA agent, and this cookbook exercises all of them:

- **Operational data store.** The canonical records (here, transactions) live in MongoDB,
  and the agent reads and updates them through audited tools.
- **Retrieval engine.** Semantic recall via `$vectorSearch`, exact matching on names, ids,
  and codes via `$search`, and a hybrid of both fused with reciprocal rank fusion, so the
  agent grounds its reasoning in real precedent instead of guessing.
- **Graph traversal.** `$graphLookup` walks relationships in place (shared accounts,
  circular flows) and surfaces network signals a flat lookup would miss.
- **Secure transactional persistence and audit.** Decisions and an append-only audit trail
  are written back to the same cluster, so every action the agent takes is durable and
  reviewable.

MongoDB here is your **operational data store and retrieval engine**: your system of record,
controlled by your application. That is distinct from CMA's own agent memory (Memory
Stores), which Claude manages on the platform. The two are complementary, not the same thing.

**By the end of this cookbook, you'll be able to:**

- Wire a credential-safe MongoDB Atlas data path into a Claude Managed Agent, three ways
- Lift vector, full-text, hybrid, and graph retrieval into an agent's custom tools
- Gate risky agent decisions behind CMA's native `requires_action` human-in-the-loop pause
- Verify **AP2 (Agent Payments Protocol)** mandates (signed JWTs proving the user
  authorized an agent-initiated payment) and make one MongoDB Atlas cluster the agent's
  system of record and audit backbone

The cookbook is one continuous build, in three sections:

1. **Connect.** Three ways to get a MongoDB client into a CMA agent's reach, and where the
   credential lives on each path.
2. **Retrieve.** The four retrieval patterns that matter for an agent (vector, full-text,
   hybrid, and graph) as runnable, liftable building blocks.
3. **The end-to-end agent.** A human-in-the-loop **fraud-review agent** that verifies each
   case's AP2 mandate chain, uses every pattern behind a human gate, and treats
   MongoDB Atlas as the system of record and audit backbone.

Payments is a natural fit for the worked example, but the patterns are vertical-agnostic.
Swap the collection and the tools, and the same shape serves a support, research, or
operations agent. The architecture diagram in Section 3 shows exactly where data and
credentials live on each path.

## Prerequisites

**Required knowledge:** Python and `pymongo` basics, plus passing familiarity with the
Claude API. New to MongoDB and Claude? The
[library-RAG notebook](../third_party/MongoDB/rag_using_mongodb.ipynb) covers the
`$vectorSearch` basics before you wire MongoDB into an agent. New to the custom-tool gate?
[`CMA_gate_human_in_the_loop.ipynb`](CMA_gate_human_in_the_loop.ipynb) teaches the round-trip
and the `requires_action` bounce without the data path.

**Required tools:** Python 3.11+, an [Anthropic API key](https://console.anthropic.com), and
a MongoDB Atlas cluster.

## Setup

**Required:** `ANTHROPIC_API_KEY` and `MONGO_URI` (a MongoDB Atlas SRV connection string). An M0 cluster works for everything except native `$rankFusion`, which requires an
M10+ cluster on MongoDB 8.1+. On older servers the cookbook automatically falls back to a
portable client-side RRF. `pymongo` and `voyageai` are already project dependencies:

```bash
uv sync --all-extras   # from the repo root
cp .env.example .env    # then add ANTHROPIC_API_KEY and MONGO_URI
```

**Optional embeddings and rerank provider.** The seed fixture ships precomputed embeddings,
so the cookbook runs without one. To enable the live-embedding and reranker paths in
Section 3, set **one** of `MDB_ATLAS_API_KEY` (the MongoDB Atlas AI endpoint, which serves
both `/v1/embeddings` and `/v1/rerank`) or `VOYAGE_API_KEY` (the `voyageai` SDK directly).
If both are set, `MDB_ATLAS_API_KEY` wins. The `provider=` field in the status line below
reflects which one was found.

**Optional toggles.** `ENABLE_RERANK=1` adds the reranker (`rerank-2.5`) second stage, which
needs a provider key from above. `AUTO_APPROVE=1` resolves Section 3's human gate
deterministically for unattended or CI runs instead of prompting inline. `COOKBOOK_MODEL`
overrides the agent model; it defaults to `claude-haiku-4-5`, fast and low-cost for this
demo, and you can set `COOKBOOK_MODEL=claude-sonnet-4-6` for a stronger model.

### Helper functions (inline library)

All retrieval, decision, and AP2 mandate helpers are defined in the cells below, grouped by
theme so each section stays navigable. Skip ahead if you like; each section points back to
the helper it uses.

In [1]:
# ── Configuration ─────────────────────────────────────────────────────────────
from __future__ import annotations

import hashlib
import uuid
from datetime import UTC, datetime, timedelta
from typing import Any

import jwt as _jwt

RRF_K = 60
DEFAULT_MODEL = "claude-haiku-4-5"
EMBED_MODEL = "voyage-4-large"
EMBED_DIM = 1024
RERANK_MODEL = "rerank-2.5"
RERANK_FANOUT = 5
CONFIDENCE_THRESHOLD_APPROVE = 85
CONFIDENCE_BAND = 10
HIGH_VALUE_LIMIT = 50_000
STRUCTURING_LOW = 4900
STRUCTURING_HIGH = 4999
DECIDED_STATUSES = ("approved", "rejected", "escalated", "completed")
VECTOR_INDEX_NAME = "transactions_vector_index"
SEARCH_INDEX_NAME = "transactions_search_index"
LEXICAL_PATHS = ["text", "sender.name", "recipient.name"]
_PROJECT_FIELDS = ["transaction_id", "text", "amount", "sender", "recipient", "decision"]
REQUIRED_ENV = ("ANTHROPIC_API_KEY", "MONGO_URI")
SEED_LANES = ("clean_approve", "clear_reject", "structuring", "high_value", "ring")

ATLAS_EMBEDDINGS_URL = "https://ai.mongodb.com/v1/embeddings"
ATLAS_RERANK_URL = "https://ai.mongodb.com/v1/rerank"


class EmbeddingDimensionMismatch(RuntimeError): ...


class IndexTimeout(RuntimeError): ...


def _parse_major_minor(version: str) -> tuple[int, int]:
    parts = str(version).split(".")
    try:
        return (int(parts[0]), int(parts[1]) if len(parts) > 1 else 0)
    except (ValueError, IndexError):
        return (0, 0)


def supports_rank_fusion(version: str) -> bool:
    return _parse_major_minor(version) >= (8, 1)

In [2]:
# ── Retrieval pipeline builders ────────────────────────────────────────────────


def _project_stage(with_score: bool = False) -> dict:
    proj: dict[str, Any] = {f: 1 for f in _PROJECT_FIELDS}
    proj["_id"] = 0
    if with_score:
        proj["score"] = {"$meta": "score"}
    return {"$project": proj}


def build_rank_fusion_pipeline(
    qvec,
    query,
    *,
    k,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    w_v=0.5,
    w_l=0.5,
    status_in=DECIDED_STATUSES,
) -> list[dict]:
    candidates = max(50, k * 10)
    per_branch = max(k * 4, 20)
    return [
        {
            "$rankFusion": {
                "input": {
                    "pipelines": {
                        "vector": [
                            {
                                "$vectorSearch": {
                                    "index": vector_index,
                                    "path": "embedding",
                                    "queryVector": qvec,
                                    "numCandidates": candidates,
                                    "limit": per_branch,
                                    "filter": {"status": {"$in": list(status_in)}},
                                }
                            }
                        ],
                        "lexical": [
                            {
                                "$search": {
                                    "index": search_index,
                                    "text": {"query": query, "path": LEXICAL_PATHS},
                                }
                            },
                            {"$limit": per_branch},
                        ],
                    }
                },
                "combination": {"weights": {"vector": w_v, "lexical": w_l}},
            }
        },
        {"$limit": k},
        _project_stage(with_score=True),
    ]


def build_vector_pipeline(
    qvec, *, limit, candidates=None, vector_index=VECTOR_INDEX_NAME, status_in=DECIDED_STATUSES
) -> list[dict]:
    return [
        {
            "$vectorSearch": {
                "index": vector_index,
                "path": "embedding",
                "queryVector": qvec,
                "numCandidates": candidates or max(50, limit * 10),
                "limit": limit,
                "filter": {"status": {"$in": list(status_in)}},
            }
        },
        _project_stage(),
    ]


def build_lexical_pipeline(query, *, limit, search_index=SEARCH_INDEX_NAME) -> list[dict]:
    return [
        {"$search": {"index": search_index, "text": {"query": query, "path": LEXICAL_PATHS}}},
        {"$limit": limit},
        _project_stage(),
    ]


def fuse_rrf(vector_hits, lexical_hits, *, w_v, w_l, k, rrf_k=RRF_K) -> list[dict]:
    scores: dict[str, float] = {}
    docs: dict[str, dict] = {}
    for hits, weight in ((vector_hits, w_v), (lexical_hits, w_l)):
        for rank, doc in enumerate(hits, start=1):
            doc_id = str(doc.get("transaction_id", ""))
            if not doc_id:
                continue
            scores[doc_id] = scores.get(doc_id, 0.0) + weight / (rrf_k + rank)
            docs.setdefault(doc_id, doc)
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)[:k]
    return [{**docs[i], "score": s} for i, s in ranked]


def rerank_pool_size(k: int, fanout: int, *, enable_rerank: bool) -> int:
    return k * fanout if enable_rerank else k


def merge_rerank(candidates, rerank_results, *, top_k) -> list[dict]:
    def _score(r):
        s = getattr(r, "relevance_score", None)
        return (
            (s if s is not None else r.get("relevance_score", float("-inf")))
            if isinstance(r, dict)
            else (s or float("-inf"))
        )

    seen: set[str] = set()
    out: list[dict] = []
    for result in sorted(rerank_results, key=_score, reverse=True):
        index = getattr(result, "index", None)
        if index is None and isinstance(result, dict):
            index = result.get("index")
        score = getattr(result, "relevance_score", None)
        if score is None and isinstance(result, dict):
            score = result.get("relevance_score")
        if index is None or index < 0 or index >= len(candidates):
            continue
        candidate = candidates[index]
        cid = str(candidate.get("transaction_id", index))
        if cid in seen:
            continue
        seen.add(cid)
        out.append({**candidate, "score": score})
        if len(out) >= top_k:
            break
    return out

In [3]:
# ── Graph traversal + escalation logic ────────────────────────────────────────


def build_graph_pipeline(
    account_id: str, *, max_depth: int = 4, collection: str = "transactions"
) -> list[dict]:
    return [
        {"$match": {"sender.account_number": account_id}},
        {
            "$graphLookup": {
                "from": collection,
                "startWith": "$recipient.account_number",
                "connectFromField": "recipient.account_number",
                "connectToField": "sender.account_number",
                "as": "chain",
                "maxDepth": max_depth,
                "depthField": "depth",
            }
        },
    ]


def summarize_ring(graph_doc: dict, *, seed_account: str) -> dict:
    chain: list[dict] = list(graph_doc.get("chain", []))
    accounts: set[str] = set()
    small_transfers = 0
    circular_flow = False
    for edge in chain:
        sender = (edge.get("sender") or {}).get("account_number")
        recipient = (edge.get("recipient") or {}).get("account_number")
        accounts.update(a for a in (sender, recipient) if a)
        if recipient == seed_account:
            circular_flow = True
        if float(edge.get("amount", 0) or 0) < 1000:
            small_transfers += 1
    network_size = len(chain)
    layering = small_transfers >= 5
    return {
        "network_size": network_size,
        "unique_accounts": len(accounts),
        "circular_flow": circular_flow,
        "layering": layering,
        "suspicious_patterns": circular_flow or layering or network_size >= 3,
    }


def is_structuring(amount: float) -> bool:
    return STRUCTURING_LOW <= float(amount) <= STRUCTURING_HIGH


def should_escalate(
    recommendation,
    confidence,
    amount,
    ring_suspicious,
    *,
    approve_threshold=CONFIDENCE_THRESHOLD_APPROVE,
    band=CONFIDENCE_BAND,
    high_value_limit=HIGH_VALUE_LIMIT,
) -> tuple[bool, str | None]:
    reasons: list[str] = []
    if (approve_threshold - band) <= confidence < approve_threshold:
        reasons.append("medium_confidence")
    if recommendation == "approve" and float(amount) >= high_value_limit:
        reasons.append("high_value")
    if is_structuring(amount):
        reasons.append("structuring")
    if ring_suspicious:
        reasons.append("fraud_ring")
    return (bool(reasons), ", ".join(reasons) if reasons else None)


def resolve_human_decision(recommendation, *, auto_approve, override_ids, txn_id) -> str:
    opposite = "reject" if recommendation == "approve" else "approve"
    if auto_approve and txn_id in set(override_ids):
        return opposite
    return recommendation

In [4]:
# ── Embedding client (MongoDB Atlas AI or Voyage SDK) ──────────────────────────────────


class _EmbedResponse:
    def __init__(self, embeddings):
        self.embeddings = embeddings


class _RerankResult:
    def __init__(self, index, relevance_score):
        self.index = index
        self.relevance_score = relevance_score


class _RerankResponse:
    def __init__(self, results):
        self.results = results


class AtlasEmbeddingClient:
    # Adapts the MongoDB Atlas AI endpoint to the voyageai .embed()/.rerank() interface.
    def __init__(
        self,
        api_key,
        *,
        url=ATLAS_EMBEDDINGS_URL,
        rerank_url=ATLAS_RERANK_URL,
        output_dimension=EMBED_DIM,
        transport=None,
    ):
        self._api_key = api_key
        self._url = url
        self._rerank_url = rerank_url
        self._dim = output_dimension
        self._transport = transport

    def _post(self, url, payload):
        if self._transport is not None:
            return self._transport(url, payload, self._api_key)
        import requests

        resp = requests.post(
            url,
            json=payload,
            headers={
                "Authorization": f"Bearer {self._api_key}",
                "Content-Type": "application/json",
            },
            timeout=30,
        )
        resp.raise_for_status()
        return resp.json()

    def embed(self, texts, model=None, input_type=None):
        body = self._post(
            self._url,
            {
                "model": model,
                "input": list(texts),
                "input_type": input_type,
                "output_dimension": self._dim,
            },
        )
        return _EmbedResponse([item["embedding"] for item in body["data"]])

    def rerank(self, query, documents, model=None, top_k=None):
        body = self._post(
            self._rerank_url,
            {"model": model, "query": query, "documents": list(documents), "top_k": top_k},
        )
        return _RerankResponse(
            [_RerankResult(d["index"], d["relevance_score"]) for d in body["data"]]
        )


def check_embedding_dim(vec, expected):
    if len(vec) != expected:
        raise EmbeddingDimensionMismatch(
            f"embedding has {len(vec)} dimensions, expected {expected}"
        )
    return vec


def embed_query(text, *, client, model=EMBED_MODEL, dim=EMBED_DIM) -> list[float]:
    result = client.embed([text], model=model, input_type="query")
    return check_embedding_dim(list(result.embeddings[0]), dim)


def embed_documents(texts, *, client, model=EMBED_MODEL, dim=EMBED_DIM) -> list[list[float]]:
    result = client.embed(list(texts), model=model, input_type="document")
    return [check_embedding_dim(list(v), dim) for v in result.embeddings]


def rerank(query, documents, *, client, model=RERANK_MODEL, top_k):
    return client.rerank(query, documents, model=model, top_k=top_k).results


def make_embedding_client(env=None):
    import os

    source = env if env is not None else os.environ
    if source.get("MDB_ATLAS_API_KEY"):
        return AtlasEmbeddingClient(source["MDB_ATLAS_API_KEY"])
    if source.get("VOYAGE_API_KEY"):
        import voyageai

        return voyageai.Client()
    return None

In [5]:
# ── Tool handlers + document builders ─────────────────────────────────────────


def _new_id(prefix):
    return f"{prefix}_{uuid.uuid4().hex[:16]}"


def _now():
    return datetime.now(UTC)


def _jsonable(value):
    if isinstance(value, dict):
        return {k: _jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(v) for v in value]
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


def _project(candidate):
    keep = ("transaction_id", "text", "amount", "sender", "recipient", "decision", "score")
    return _jsonable({k: candidate[k] for k in keep if k in candidate})


def build_decision_doc(
    transaction_id,
    decision,
    *,
    confidence,
    risk_factors,
    reasoning,
    reviewed_by,
    decision_id=None,
    created_at=None,
) -> dict:
    return {
        "decision_id": decision_id or _new_id("dec"),
        "transaction_id": transaction_id,
        "decision": decision,
        "confidence_score": confidence,
        "risk_factors": list(risk_factors),
        "reasoning": reasoning,
        "reviewed_by": reviewed_by,
        "created_at": created_at or _now(),
    }


def build_audit_event(
    event_type,
    transaction_id,
    *,
    decision_id=None,
    severity="info",
    event_data=None,
    event_id=None,
    timestamp=None,
) -> dict:
    return {
        "event_id": event_id or _new_id("evt"),
        "timestamp": timestamp or _now(),
        "event_type": event_type,
        "transaction_id": transaction_id,
        "decision_id": decision_id,
        "severity": severity,
        "event_data": event_data or {},
    }


def tool_get_transaction(coll, transaction_id) -> dict:
    doc = coll.find_one({"transaction_id": transaction_id})
    if not doc:
        return {"error": "not_found", "transaction_id": transaction_id}
    return _jsonable({k: v for k, v in doc.items() if k not in ("embedding", "_id")})


def tool_hybrid_search_similar_frauds(
    coll,
    transaction_id,
    k,
    *,
    use_rank_fusion,
    enable_rerank=False,
    fanout=RERANK_FANOUT,
    reranker=None,
    vector_index=VECTOR_INDEX_NAME,
    search_index=SEARCH_INDEX_NAME,
    w_v=0.5,
    w_l=0.5,
    status_in=DECIDED_STATUSES,
) -> dict:
    txn = coll.find_one({"transaction_id": transaction_id})
    if not txn:
        return {"error": "not_found", "transaction_id": transaction_id, "similar": []}
    qvec = txn["embedding"]
    query = txn.get("text", "")
    pool_k = rerank_pool_size(k, fanout, enable_rerank=enable_rerank)
    if use_rank_fusion:
        candidates = list(
            coll.aggregate(
                build_rank_fusion_pipeline(
                    qvec,
                    query,
                    k=pool_k,
                    vector_index=vector_index,
                    search_index=search_index,
                    w_v=w_v,
                    w_l=w_l,
                    status_in=status_in,
                )
            )
        )
    else:
        vec = list(
            coll.aggregate(
                build_vector_pipeline(
                    qvec, limit=pool_k, vector_index=vector_index, status_in=status_in
                )
            )
        )
        lex = list(
            coll.aggregate(build_lexical_pipeline(query, limit=pool_k, search_index=search_index))
        )
        candidates = fuse_rrf(vec, lex, w_v=w_v, w_l=w_l, k=pool_k)
    candidates = [c for c in candidates if str(c.get("transaction_id")) != str(transaction_id)]
    if enable_rerank and reranker is not None and candidates:
        results = reranker(query, [c.get("text", "") for c in candidates], top_k=k)
        candidates = merge_rerank(candidates, results, top_k=k)
    else:
        candidates = candidates[:k]
    return {"similar": [_project(c) for c in candidates]}


def tool_detect_fraud_ring(coll, account_id, *, max_depth=4) -> dict:
    pipeline = build_graph_pipeline(account_id, max_depth=max_depth, collection=coll.name)
    docs = list(coll.aggregate(pipeline))
    return _jsonable(summarize_ring(docs[0] if docs else {"chain": []}, seed_account=account_id))


def tool_record_decision(
    db,
    transaction_id,
    decision,
    *,
    confidence,
    risk_factors,
    reasoning,
    reviewed_by,
    escalated=False,
    recommended_decision=None,
) -> dict:
    decision_doc = build_decision_doc(
        transaction_id,
        decision,
        confidence=confidence,
        risk_factors=risk_factors,
        reasoning=reasoning,
        reviewed_by=reviewed_by,
    )
    db["transaction_decisions"].insert_one(decision_doc)
    if escalated:
        audit = build_audit_event(
            "escalated_to_human",
            transaction_id,
            decision_id=decision_doc["decision_id"],
            severity="warning",
            event_data={"human_decision": decision, "recommended_decision": recommended_decision},
        )
    else:
        audit = build_audit_event(
            "decision_stored", transaction_id, decision_id=decision_doc["decision_id"]
        )
    db["audit_events"].insert_one(audit)
    db["transactions"].update_one(
        {"transaction_id": transaction_id}, {"$set": {"status": decision}}
    )
    return {"recorded": True, "decision_id": decision_doc["decision_id"]}

In [6]:
# ── Gate loop + env/MongoDB Atlas helpers ─────────────────────────────────────────────


def next_unanswered_event_ids(stop_reason, already_responded) -> list[str]:
    ids = getattr(stop_reason, "event_ids", None) or (
        stop_reason.get("event_ids") if isinstance(stop_reason, dict) else None
    )
    return [e for e in (ids or []) if e not in already_responded]


def run_gate_loop(stream, send, *, handlers, resolver) -> dict:
    pending: dict[str, Any] = {}
    responded: set[str] = set()
    serviced: list[tuple[str, dict]] = []
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            pending[ev.id] = ev
        elif ev.type == "session.status_idle":
            stop = getattr(ev, "stop_reason", None)
            if stop is None:
                continue
            if stop.type != "requires_action":
                break
            for event_id in next_unanswered_event_ids(stop, responded):
                call = pending.get(event_id)
                if call is None:
                    continue
                if call.name == "escalate":
                    result = {"human_decision": resolver(call.input)}
                else:
                    handler = handlers.get(call.name)
                    result = (
                        handler(call.input) if handler else {"error": f"unknown tool {call.name}"}
                    )
                responded.add(event_id)
                send(event_id, result)
                serviced.append((call.name, result))
        elif ev.type == "session.status_terminated":
            break
    return {"serviced": serviced, "responded": sorted(responded)}


def resolve_model(env=None) -> str:
    import os

    return (env if env is not None else os.environ).get("COOKBOOK_MODEL", DEFAULT_MODEL)


def missing_required_env(env=None) -> list[str]:
    import os

    source = env if env is not None else os.environ
    return [name for name in REQUIRED_ENV if not source.get(name)]


def server_version(coll) -> str:
    return str(coll.database.command("buildInfo").get("version", "0.0"))


def prepare_seed(docs, *, now) -> list[dict]:
    prepared = []
    for doc in docs:
        doc = dict(doc)
        days = doc.pop("created_days_ago", 0)
        doc["created_at"] = now - timedelta(days=days)
        prepared.append(doc)
    return prepared


def seed_collection(coll, docs) -> int:
    coll.delete_many({})
    if docs:
        coll.insert_many(list(docs))
    return coll.count_documents({})


def ensure_indexes(
    coll, *, dim=EMBED_DIM, poll_interval=2.0, timeout_s=180.0, sleep=None, monotonic=None
) -> None:
    import time

    sleep = sleep or time.sleep
    monotonic = monotonic or time.monotonic
    coll.create_index("transaction_id", unique=True)
    existing = {idx["name"] for idx in coll.list_search_indexes()}
    if VECTOR_INDEX_NAME not in existing:
        coll.create_search_index(vector_index_model(dim))
    if SEARCH_INDEX_NAME not in existing:
        coll.create_search_index(search_index_model())
    deadline = monotonic() + timeout_s
    names = (VECTOR_INDEX_NAME, SEARCH_INDEX_NAME)
    while True:
        status = {idx["name"]: idx for idx in coll.list_search_indexes()}
        if all(status.get(n, {}).get("queryable") is True for n in names):
            return
        if monotonic() >= deadline:
            pending = [n for n in names if status.get(n, {}).get("queryable") is not True]
            raise IndexTimeout(f"indexes not queryable within {timeout_s}s: {pending}")
        sleep(poll_interval)


def vector_index_model(dim=EMBED_DIM) -> dict:
    return {
        "name": VECTOR_INDEX_NAME,
        "type": "vectorSearch",
        "definition": {
            "fields": [
                {
                    "type": "vector",
                    "path": "embedding",
                    "numDimensions": dim,
                    "similarity": "cosine",
                },
                {"type": "filter", "path": "status"},
            ]
        },
    }


def search_index_model() -> dict:
    return {
        "name": SEARCH_INDEX_NAME,
        "type": "search",
        "definition": {"mappings": {"dynamic": True}},
    }


def preflight(coll, *, expected_dim=EMBED_DIM) -> dict:
    issues: list[str] = []
    try:
        coll.database.command("ping")
    except Exception as exc:
        return {
            "ok": False,
            "issues": [f"cannot reach MongoDB (check MONGO_URI / IP allowlist): {exc}"],
        }
    indexes = {idx["name"]: idx for idx in coll.list_search_indexes()}
    for name in (VECTOR_INDEX_NAME, SEARCH_INDEX_NAME):
        index = indexes.get(name)
        if index is None:
            issues.append(
                f"missing MongoDB Atlas Search index '{name}' — create it, then wait until queryable"
            )
        elif index.get("queryable") is not True:
            issues.append(f"index '{name}' is still building — wait until it is queryable")
    vector_index = indexes.get(VECTOR_INDEX_NAME)
    if vector_index is not None:
        definition = vector_index.get("latestDefinition") or vector_index.get("definition") or {}
        if not any(
            f.get("type") == "filter" and f.get("path") == "status"
            for f in definition.get("fields", [])
        ):
            issues.append(
                "vector index is missing a `status` filter field — recreate from vector_index_model()"
            )
    return {"ok": not issues, "issues": issues}

In [7]:
# ── AP2 mandate helpers ────────────────────────────────────────────────────────


class MandateVerificationError(ValueError): ...


def build_mandate_content(
    mandate_type, agent_pk, mandate_id, constraints, checkout_mandate_hash=None
) -> dict:
    content: dict = {
        "mandate_type": mandate_type,
        "mandate_id": mandate_id,
        "agent_pk": agent_pk,
        "constraints": dict(constraints),
    }
    if checkout_mandate_hash is not None:
        content["checkout_mandate_hash"] = checkout_mandate_hash
    return content


def sign_mandate_jwt(content, private_key, *, exp_delta=None) -> str:
    now = datetime.now(UTC)
    delta = exp_delta if exp_delta is not None else timedelta(hours=24)
    payload = {
        "sub": content["mandate_type"],
        "iat": int(now.timestamp()),
        "exp": int((now + delta).timestamp()),
        **content,
    }
    return _jwt.encode(payload, private_key, algorithm="ES256")


def verify_mandate_jwt(token, public_key) -> dict:
    try:
        return _jwt.decode(token, public_key, algorithms=["ES256"])
    except _jwt.InvalidTokenError as exc:
        raise MandateVerificationError(str(exc)) from exc


def attach_mandates(coll, transaction_ids, *, agent_pk, ts_private_key) -> dict[str, dict]:
    result: dict[str, dict] = {}
    expiry = (datetime.now(UTC) + timedelta(hours=24)).isoformat()
    for txn_id in transaction_ids:
        doc = coll.find_one({"transaction_id": txn_id})
        if doc is None:
            continue
        mandate_id = _new_id("mandate")
        checkout_content = build_mandate_content(
            mandate_type="checkout",
            agent_pk=agent_pk,
            mandate_id=mandate_id,
            constraints={
                "max_amount": doc["amount"] * 1.1,
                "currency": doc.get("currency", "USD"),
                "transaction_type": doc.get("transaction_type", "purchase"),
                "not_after": expiry,
            },
        )
        checkout_jwt = sign_mandate_jwt(checkout_content, ts_private_key)
        checkout_hash = hashlib.sha256(checkout_jwt.encode()).hexdigest()
        payment_content = build_mandate_content(
            mandate_type="payment",
            agent_pk=agent_pk,
            mandate_id=mandate_id,
            constraints={"max_amount": doc["amount"] * 1.1, "currency": doc.get("currency", "USD")},
            checkout_mandate_hash=checkout_hash,
        )
        payment_jwt = sign_mandate_jwt(payment_content, ts_private_key)
        coll.update_one(
            {"transaction_id": txn_id},
            {
                "$set": {
                    "checkout_mandate_jwt": checkout_jwt,
                    "payment_mandate_jwt": payment_jwt,
                    "mandate_id": mandate_id,
                    "agent_pk": agent_pk,
                }
            },
        )
        result[txn_id] = {
            "checkout_mandate_jwt": checkout_jwt,
            "payment_mandate_jwt": payment_jwt,
            "mandate_id": mandate_id,
            "agent_pk": agent_pk,
        }
    return result


def tool_verify_mandates(db, transaction_id, ts_public_key) -> dict:
    errors: list[str] = []
    coll = db["transactions"]
    doc = coll.find_one({"transaction_id": transaction_id})
    if doc is None:
        return {
            "valid": False,
            "constraints_satisfied": False,
            "double_spend_detected": False,
            "agent_pk": None,
            "mandate_id": None,
            "errors": ["transaction_not_found"],
        }
    checkout_jwt = doc.get("checkout_mandate_jwt")
    payment_jwt = doc.get("payment_mandate_jwt")
    if not checkout_jwt or not payment_jwt:
        return {
            "valid": False,
            "constraints_satisfied": False,
            "double_spend_detected": False,
            "agent_pk": doc.get("agent_pk"),
            "mandate_id": doc.get("mandate_id"),
            "errors": ["mandate_missing"],
        }
    checkout_payload: dict | None = None
    try:
        checkout_payload = verify_mandate_jwt(checkout_jwt, ts_public_key)
    except MandateVerificationError:
        errors.append("signature_invalid")
    payment_payload: dict | None = None
    try:
        payment_payload = verify_mandate_jwt(payment_jwt, ts_public_key)
    except MandateVerificationError:
        if "signature_invalid" not in errors:
            errors.append("signature_invalid")
    if checkout_payload is not None and payment_payload is not None:
        expected_hash = hashlib.sha256(checkout_jwt.encode()).hexdigest()
        if payment_payload.get("checkout_mandate_hash", "") != expected_hash:
            errors.append("checkout_hash_mismatch")
    if errors:
        return {
            "valid": False,
            "constraints_satisfied": False,
            "double_spend_detected": False,
            "agent_pk": doc.get("agent_pk"),
            "mandate_id": doc.get("mandate_id"),
            "errors": errors,
        }
    constraints_ok = True
    if checkout_payload:
        c = checkout_payload.get("constraints", {})
        if doc["amount"] > c.get("max_amount", float("inf")):
            errors.append("amount_exceeded")
            constraints_ok = False
        doc_type = doc.get("transaction_type", "")
        c_type = c.get("transaction_type", "")
        if c_type and doc_type != c_type:
            errors.append("type_mismatch")
            constraints_ok = False
        not_after_str = c.get("not_after")
        if not_after_str:
            try:
                if datetime.now(UTC) > datetime.fromisoformat(not_after_str):
                    errors.append("mandate_expired")
                    constraints_ok = False
            except ValueError:
                errors.append("mandate_expired")
                constraints_ok = False
    mandate_id = (checkout_payload or {}).get("mandate_id") or doc.get("mandate_id")
    agent_pk = (checkout_payload or {}).get("agent_pk") or doc.get("agent_pk")
    prior = db["mandate_receipts"].find_one({"mandate_id": mandate_id, "agent_pk": agent_pk})
    return {
        "valid": True,
        "constraints_satisfied": constraints_ok,
        "double_spend_detected": prior is not None,
        "agent_pk": agent_pk,
        "mandate_id": mandate_id,
        "errors": errors,
    }


def store_mandate_receipt(db, mandate_id, agent_pk, checkout_mandate_hash, decision) -> None:
    db["mandate_receipts"].insert_one(
        {
            "mandate_id": mandate_id,
            "agent_pk": agent_pk,
            "checkout_mandate_hash": checkout_mandate_hash,
            "decision": decision,
            "timestamp": _now(),
        }
    )

In [8]:
# Pre-computed fixture: 20 fraud-review seed transactions with 1024-dim Voyage embeddings.
# Embedded as gzip+base64 so the cookbook data seed runs without an embedding API key.
import base64 as _b64
import gzip as _gz
import json as _json

_FIXTURE_GZ_B64 = (
    b"..."
    b"21Uu//P//qf/Bxzbrq02WAQA"
)

_fixture_jsonl = _gz.decompress(_b64.b64decode(_FIXTURE_GZ_B64)).decode()
SEED_DOCS_RAW = [_json.loads(line) for line in _fixture_jsonl.splitlines() if line.strip()]
print(f"fixture: {len(SEED_DOCS_RAW)} transactions loaded")

fixture: 20 transactions loaded


With the helpers and fixture in place, load the environment, connect the Claude and
MongoDB Atlas clients, and resolve the run configuration. The status line confirms the
model, the rerank toggle, and which embedding provider (if any) was found.

In [9]:
import json
import os
import sys
from datetime import UTC, datetime
from pathlib import Path

import dotenv
from anthropic import Anthropic
from pymongo import MongoClient

sys.path.insert(0, str(Path.cwd().parent))
from utilities import wait_for_idle_status

dotenv.load_dotenv()

missing = missing_required_env()
assert not missing, f"Set these in your environment / .env: {missing}"

MODEL = resolve_model()
ENABLE_RERANK = os.getenv("ENABLE_RERANK", "").lower() in ("1", "true")
AUTO_APPROVE = os.getenv("AUTO_APPROVE", "").lower() in (
    "1",
    "true",
)  # non-interactive gate (for CI / unattended runs)

client = Anthropic()
mongo = MongoClient(os.environ["MONGO_URI"])
db = mongo["fraud_review_demo"]
coll = db["transactions"]

# Embeddings + rerank provider: the MongoDB Atlas AI endpoint (MDB_ATLAS_API_KEY) or the
# voyageai SDK (VOYAGE_API_KEY). None if neither is set.
ai_client = make_embedding_client()

print(
    f"model={MODEL}  rerank={'on' if ENABLE_RERANK else 'off'}  provider={'yes' if ai_client else 'none'}"
)

ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


model=claude-haiku-4-5  rerank=on  provider=yes


## Section 1. Connect: getting MongoDB into a managed agent

The agent loop and its sandbox are **Anthropic-hosted**. That is what "Managed" means. The
only thing you host is the **data path**: a process that holds `pymongo`, a sandbox image you
build, or a MongoDB MCP server you stand up. That gives you three connection paths. On every
one, the MongoDB credential lives on *your* side of the boundary. It is never placed in the
agent context, a cloud sandbox's environment, or a file the agent can read.

All three paths follow from that one constraint, so pick by deployment shape, not security.

### Path A: host-side custom tool (recommended)

Your application defines a `custom` tool. When the agent calls it, the session pauses with
`requires_action`, **your** process runs the query with `pymongo`, and you post back a
`user.custom_tool_result`. The secret never enters the sandbox, MongoDB can sit in a private
VPC, and the agent is constrained to a fixed, audited set of queries. It is also the lightest
path: there is no standing server to host. The orchestrator (your app, a Lambda, or this
notebook) only needs outbound HTTPS to the Claude API and to MongoDB. This is Anthropic's
recommended pattern for any secret-bearing data source.

To see the wiring in isolation, we use a deliberately tiny example: a throwaway `notes`
collection and one `find_notes` tool, so the round-trip shape is unobscured. Section 3 scales
this exact shape to a real workload (five tools, hybrid retrieval, a human gate). The sandbox
needs no network, because the data path is the host-side tool round-trip.

In [10]:
notes = mongo["cookbook_mongodb_on_cma"]["notes"]
notes.delete_many({})
notes.insert_many(
    [
        {"note_id": "n1", "tag": "todo", "text": "Renew the SSL certificate before it expires."},
        {"note_id": "n2", "tag": "idea", "text": "Add a dark-mode toggle to the dashboard."},
        {"note_id": "n3", "tag": "todo", "text": "Email the quarterly report to the finance team."},
        {"note_id": "n4", "tag": "idea", "text": "Prototype a CLI for the export pipeline."},
        {"note_id": "n5", "tag": "todo", "text": "Rotate the staging database credentials."},
    ]
)

notes_agent = client.beta.agents.create(
    name="cookbook-mongodb-notes",
    model=MODEL,
    system=(
        "You answer questions about the user's notes. Use the find_notes tool to fetch "
        "notes (optionally filtered by `tag`) before answering. Make as many find_notes "
        "calls as you need, then give a concise final answer and stop."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {"enabled": True, "permission_policy": {"type": "always_allow"}},
        },
        {
            "type": "custom",
            "name": "find_notes",
            "description": "Return stored notes, optionally filtered by tag (e.g. 'todo', 'idea').",
            "input_schema": {
                "type": "object",
                "properties": {"tag": {"type": "string", "description": "optional tag filter"}},
                "required": [],
            },
        },
    ],
)

notes_env = client.beta.environments.create(
    name="cookbook-mongodb-notes-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

notes_session = client.beta.sessions.create(
    environment_id=notes_env.id,
    agent={"type": "agent", "id": notes_agent.id, "version": notes_agent.version},
    title="MongoDB notes (host-side custom tool)",
)
print(f"seeded {notes.count_documents({})} notes; session: {notes_session.id}")

seeded 5 notes; session: sesn_01GzLpWSzLJ5PrgCmbhyrTsN


The agent now has the `find_notes` tool. We open the event stream, ask a question, and drive
the round-trip: on each `agent.custom_tool_use`, run the query **host-side** and post a
`user.custom_tool_result`, then break when the session idles with `end_turn`. Two details
matter:

- Results are projected with `{"_id": 0}` because a raw MongoDB `ObjectId` is not
  JSON-serializable, so it must not go into a `custom_tool_result`.
- We dedupe `event_ids` across `status_idle` events because the server returns the pending
  tool calls as a sliding window. The
  [gate notebook](CMA_gate_human_in_the_loop.ipynb) documents the same pattern.

In [11]:
def run_find_notes(args: dict) -> dict:
    """Host-side handler: pymongo runs here; _id (ObjectId) is projected out so the result is JSON-able."""
    flt = {"tag": args["tag"]} if args.get("tag") else {}
    return {"notes": list(notes.find(flt, {"_id": 0}))}


tool_use_events = {}
responded = set()
round_trips = []  # record each call so we can show the host-side round-trip
question = "Using find_notes, list every note tagged 'todo', then tell me how many there are."

with client.beta.sessions.events.stream(notes_session.id) as stream:
    client.beta.sessions.events.send(
        session_id=notes_session.id,
        events=[{"type": "user.message", "content": [{"type": "text", "text": question}]}],
    )
    for ev in stream:
        if ev.type == "agent.custom_tool_use":
            tool_use_events[ev.id] = ev
        elif ev.type == "session.status_idle" and ev.stop_reason:
            if ev.stop_reason.type == "requires_action":
                for event_id in ev.stop_reason.event_ids:
                    if event_id in responded:
                        continue
                    tev = tool_use_events[event_id]
                    if tev.name == "find_notes":
                        result = run_find_notes(tev.input)
                        round_trips.append(
                            {"tool_input": tev.input, "returned": len(result["notes"])}
                        )
                    else:
                        result = {"error": f"unknown tool {tev.name}"}
                    client.beta.sessions.events.send(
                        session_id=notes_session.id,
                        events=[
                            {
                                "type": "user.custom_tool_result",
                                "custom_tool_use_id": event_id,
                                "content": [{"type": "text", "text": json.dumps(result)}],
                            }
                        ],
                    )
                    responded.add(event_id)
            elif ev.stop_reason.type == "end_turn":
                break
        elif ev.type == "session.status_terminated":
            break

wait_for_idle_status(client, notes_session.id)

print("host-side find_notes round-trips:")
for rt in round_trips:
    print(f"  find_notes({rt['tool_input']}) -> {rt['returned']} notes")

client.beta.sessions.archive(notes_session.id)
client.beta.environments.archive(notes_env.id)
client.beta.agents.archive(notes_agent.id)
notes.drop()
print("archived the demo agent / environment / session; dropped the notes collection")

host-side find_notes round-trips:
  find_notes({'tag': 'todo'}) -> 3 notes
archived the demo agent / environment / session; dropped the notes collection


### Path B: self-hosted sandbox, in-sandbox pymongo

CMA can also run the **sandbox** on infrastructure you control. The agent loop stays on
Anthropic, but the per-session container is one you build and run. Because you own the image,
the MongoDB client can live *inside* the sandbox, and the connection string is a normal
**environment variable** that never enters a prompt, the control plane, or the session event
history. The agent then queries MongoDB straight from its `bash` tool:

```sh
python3 -c 'import os; from pymongo import MongoClient; \
c = MongoClient(os.environ["MONGODB_URI"]); \
print(list(c["mydb"]["mycoll"].find({}, {"_id": 0}).limit(5)))'
```

The complete, runnable variant lives in
[`self_hosted_sandboxes/docker/`](self_hosted_sandboxes/docker/README.md). The
[`Dockerfile`](self_hosted_sandboxes/docker/Dockerfile) bundles `python3` and `pymongo` into
the per-session image, [`on-work.sh`](self_hosted_sandboxes/docker/on-work.sh) forwards an
optional `MONGODB_URI` into each per-session container, and
[`start.sh`](self_hosted_sandboxes/docker/start.sh) builds the image and starts the host-side
poller. Prefer the MongoDB shell? Add `mongosh` to the image (MongoDB's apt repo) and call
`mongosh "$MONGODB_URI" --eval '…'`.

Two caveats keep this path honest:

- **Least privilege.** The agent's `bash` runs inside that container and can read the
  environment, which is fine when you trust the task. For least privilege, swap the built-in
  toolset for your own worker-side tool that exposes a narrow `mongo_query(...)` instead of
  the raw URI.
- **Cloud sandboxes are different.** On a **cloud** sandbox there is no environment-variable
  or vault channel for a database secret (vaults are MCP-only), so a connection string handed
  in via a message or a mounted file **persists in the session event history**. You can still
  add the client through the environment `packages` config (`npm: ["mongosh"]` and/or
  `pip: ["pymongo"]`; MongoDB's apt repo isn't configured on the cloud image, so install via
  npm/pip, not apt), but use it for **public or non-secret data only**. For anything
  credential-bearing on a cloud sandbox, use Path A or Path C.

### Path C: self-hosted MongoDB MCP (until native support lands)

Host the official MongoDB MCP server behind HTTPS, then register it on the agent as an
`mcp_toolset`. This lets the agent query MongoDB flexibly across the full surface (`find`,
`aggregate`, `$vectorSearch`, admin tools) rather than the fixed query set a custom tool
exposes. This cookbook documents the shape rather than standing a server up; the wiring is
the standard CMA MCP pattern:

```python
# 1. A vault holds the bearer token for the agent -> MCP-server hop.
vault = client.beta.vaults.create(display_name="mongodb-mcp")
client.beta.vaults.credentials.create(
    vault_id=vault.id,
    display_name="MongoDB MCP",
    auth={
        "type": "static_bearer",
        "mcp_server_url": "https://mongodb-mcp.example.internal/mcp/",
        "token": MCP_BEARER_TOKEN,  # authenticates the agent to YOUR server
    },
)

# 2. The agent lists the server and exposes its tools as an mcp_toolset.
mcp_agent = client.beta.agents.create(
    name="mongodb-mcp-agent",
    model=MODEL,
    system="...",
    mcp_servers=[
        {"type": "url", "name": "mongodb", "url": "https://mongodb-mcp.example.internal/mcp/"}
    ],
    tools=[
        {
            "type": "mcp_toolset",
            "mcp_server_name": "mongodb",
            "default_config": {"enabled": True, "permission_policy": {"type": "always_allow"}},
        }
    ],
)

# 3. Sessions reference the vault; the agent never sees the token.
mcp_session = client.beta.sessions.create(
    agent={"type": "agent", "id": mcp_agent.id, "version": mcp_agent.version},
    environment_id=environment.id,
    vault_ids=[vault.id],
)
```

Two auth layers, two different secrets. The **bearer token** in the CMA vault authenticates
the agent-to-server hop, while the **MongoDB credentials** stay server-side as the MCP
server's own configuration; they never reach the vault, the agent, or the sandbox. Enforce
writes server-side by running the server **read-only** rather than relying on per-call
confirmation alone. The MCP-toolset and vault pattern is walked through in
[`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb), and
[`cma-mcp/`](cma-mcp/) documents this repo's HTTP and bearer-auth MCP conventions.

**The future path.** Native remote-MCP support and a pre-installed MongoDB client on the
cloud sandbox are on the roadmap. This cookbook will pick that path up when it lands; no date
is promised here. Until then, the three paths above are the working options.

### Which path to pick

| Path | Who runs the query | Where the secret lives | When to pick |
| --- | --- | --- | --- |
| **A. Host-side custom tool** | Your backend process | Your backend (`MONGO_URI`) | Default. Lightest path; private/VPC MongoDB Atlas; a fixed, audited query set. |
| **B. In-sandbox `pymongo`** | The agent, in the sandbox | Self-hosted: your container's env. Cloud: the session event history (unsafe) | Self-hosted sandboxes; or cloud with public / non-secret data only. |
| **C. Self-hosted remote MCP** | The agent, via the MCP server | The MCP server's config | The agent should query MongoDB flexibly across its full surface and you can host a server behind HTTPS. |

The rest of this cookbook builds on **Path A**. It is the recommended default, and it keeps
the demo self-contained: this notebook process *is* the "backend".

## Section 2. Retrieve: four patterns, one engine

An agent is only as well-grounded as its retrieval. This section shows the four MongoDB
retrieval patterns **in isolation** (here's the pipeline builder, here's what it returns) so
you can lift a single pattern into your own agent's tools. Every pipeline is built by the
same functions Section 3's agent uses in production, defined inline in the helper cells
above:

| Pattern | Builder | MongoDB stage |
| --- | --- | --- |
| Vector search | `build_vector_pipeline` | `$vectorSearch` |
| Full-text search | `build_lexical_pipeline` | `$search` |
| Hybrid (reciprocal rank fusion) | `build_rank_fusion_pipeline` / `fuse_rrf` | `$rankFusion` (8.1+) or client-side |
| Graph traversal | `build_graph_pipeline` | `$graphLookup` |

### Seed MongoDB Atlas and build the indexes (once)

We load the fixture embedded in this notebook: a small precedent corpus of already-decided
transactions plus a pending review queue, spanning every decision lane. **This one seed
serves the whole cookbook.** Section 2 queries these documents, and Section 3's agent
reviews, decides, and audits the very same ones. Seeding and index creation are idempotent,
so re-running the cookbook is clean.

`created_at` is stamped from each record's `created_days_ago`, so the seeded data stays
current relative to whenever you run this. On a fresh cluster the index build can take a
minute or two.

In [12]:
docs = prepare_seed(SEED_DOCS_RAW, now=datetime.now(UTC))
count = seed_collection(coll, docs)
print(f"seeded {count} transactions")

# Create the vector + MongoDB Atlas Search indexes if absent, then wait until both are queryable.
ensure_indexes(coll, dim=EMBED_DIM)

check = preflight(coll)
if not check["ok"]:
    for issue in check["issues"]:
        print("PREFLIGHT:", issue)
assert check["ok"], "Fix the preflight issues above before continuing."

USE_RANK_FUSION = supports_rank_fusion(server_version(coll))
print(
    f"server={server_version(coll)}  retrieval={'$rankFusion' if USE_RANK_FUSION else 'client-side RRF (fallback)'}"
)

seeded 20 transactions
server=8.3.4  retrieval=$rankFusion


### Pattern 1: Vector search (`$vectorSearch`)

Semantic recall: find documents whose `embedding` is nearest the query vector. Here we reuse
an existing document's embedding as the query (a pending case, looking for decided
precedent), so no embedding API call is needed. The builder filters to decided cases via the
`status` field, which is why the index declares `status` as a filter field.

In [13]:
query_doc = coll.find_one({"transaction_id": "txn-review-struct"})  # a pending case
hits = list(coll.aggregate(build_vector_pipeline(query_doc["embedding"], limit=3)))
print("nearest decided precedents:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

nearest decided precedents:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### Pattern 2: Full-text search (`$search`, BM25)

Keyword and phrase matching over text fields (`LEXICAL_PATHS` is the narrative plus party
names). This is what catches the exact names, ids, and codes that embeddings blur. Unlike the
vector builder above, the lexical builder applies no decided-only filter, so the pending case
itself can appear in its own results; Section 3's precedent tool filters the case under
review out.

In [14]:
hits = list(
    coll.aggregate(
        build_lexical_pipeline("cash deposit just under the reporting threshold", limit=3)
    )
)
print("full-text matches:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

full-text matches:
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-review-struct: Cash deposit of 4950 USD, just under the 5000 reporting threshold, follo
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account


### Pattern 3: Hybrid (reciprocal rank fusion)

Fuse the vector and full-text rankings. On MongoDB 8.1+ this is a single `$rankFusion`
aggregation; on 8.0 it falls back to client-side `fuse_rrf`, with the same result shape.

In [15]:
qvec = query_doc["embedding"]
query = "structuring: cash deposit just under the threshold"
if USE_RANK_FUSION:
    hits = list(coll.aggregate(build_rank_fusion_pipeline(qvec, query, k=3)))
    how = "$rankFusion (server-side)"
else:
    vec = list(coll.aggregate(build_vector_pipeline(qvec, limit=10)))
    lex = list(coll.aggregate(build_lexical_pipeline(query, limit=10)))
    hits = fuse_rrf(vec, lex, w_v=0.5, w_l=0.5, k=3)
    how = "client-side fuse_rrf (fallback)"
print(f"hybrid via {how}:")
for h in hits:
    print(f"  {h['transaction_id']}: {h['text'][:72]}")

hybrid via $rankFusion (server-side):
  txn-struct-01: Cash deposit of 4950 USD, just under the 5000 reporting threshold. Third
  txn-struct-02: Cash deposit of 4900 USD just below the 5000 CTR threshold, same account
  txn-struct-03: Transfer of 4999 USD deliberately under 5000 to avoid reporting. Pattern


### Pattern 4: Graph traversal (`$graphLookup`)

Follow `sender.account_number -> recipient.account_number` links to surface a network
(here, a circular money-flow ring). This is a *relationship* signal, deliberately separate
from the similarity ranking above.

In [16]:
graph_doc = next(
    iter(coll.aggregate(build_graph_pipeline("ACC-RING-A", collection=coll.name))), {"chain": []}
)
ring = summarize_ring(graph_doc, seed_account="ACC-RING-A")
print("graph traversal from ACC-RING-A:")
print(
    f"  network_size={ring['network_size']} circular_flow={ring['circular_flow']} suspicious={ring['suspicious_patterns']}"
)

graph traversal from ACC-RING-A:
  network_size=4 circular_flow=True suspicious=True


### Adapt these for your domain

Each builder is a generic shape; point it at your collection and fields:

- **Vector:** embed your text with any model, store it on a field, create an
  [MongoDB Atlas Vector Search index](https://www.mongodb.com/docs/atlas/atlas-vector-search/create-index/)
  on that field, and pass your query vector to `build_vector_pipeline` (or write the
  three-line `$vectorSearch` stage yourself).
- **Full-text:** set the searchable paths (`LEXICAL_PATHS`) to your text fields.
- **Hybrid:** `fuse_rrf` keys results by an id field; use your own document key.
- **Graph:** change the `connectFromField` / `connectToField` to your relationship
  (citations, org charts, supply links, follows).

Patterns in hand, Section 3 wires them into an agent: each pipeline becomes a host-side
custom tool (the Path A shape from Section 1), and the same collection becomes the agent's
system of record.

## Section 3. The end-to-end agent: human-in-the-loop fraud review

Now everything works together. This section turns a Managed Agent into a human-in-the-loop
fraud reviewer. The agent reviews flagged transactions, grounding each recommendation in the
MongoDB Atlas signals from Section 2 (exposed as **custom tools** over the Path A round-trip from
Section 1), and pauses for a human on the risky cases.

**What it demonstrates**

- **Deterministic AP2 mandate verification:** signature, constraint, and double-spend checks
  run as a hard gate before the model reasons about the case.
- **Hybrid precedent retrieval:** vector plus full-text fused with reciprocal rank fusion
  (`$rankFusion` on MongoDB 8.1+, the client-side RRF fallback otherwise), with an optional
  reranker second stage (Voyage `rerank-2.5`, served via the MongoDB Atlas AI endpoint or the Voyage
  SDK).
- **Graph ring detection:** `$graphLookup` over the sender→recipient chain, kept as a
  *separate* signal (not fused into the ranking).
- **A CMA-native human gate:** when a case is risky the session pauses (`requires_action`)
  and you, the notebook user, approve or reject. The agent then records your verdict with an
  audit trail.
- **MongoDB Atlas as the system of record:** decisions and an append-only audit trail land in the
  same cluster the agent retrieves from.

### Architecture at a glance

<p align="center">
  <img src="../images/cma_with_mongodb_atlas.jpg" width="70%" alt="Architecture diagram: the agent loop runs on Anthropic while the data path runs in your notebook; MongoDB Atlas serves the retrieval signals and stores the decision and audit trail." />
</p>

The **agent loop runs on Anthropic**; the **data path runs in your notebook**, so `MONGO_URI`
and the embedding key never enter the agent or its sandbox. MongoDB Atlas serves all three retrieval
signals and stores the decision + audit trail. Risky cases bounce to a human via `escalate`.

### What this is, and what it is not

The human-in-the-loop here is the CMA's native `requires_action` gate. The session sits
**idle on the server side** until you respond, so the *human wait* is durable. The simple
streaming loop used below, though, is meant for development: it does not survive a process
restart, and it has no retries, timeouts, or concurrency. For production, drive the same
agent from a durable backend and use the webhook pattern (end of this section) so a restart
can't drop an in-flight review.

**Bottom line:** the CMA gives you a durable *human-wait*. Durable *multi-step
orchestration* (retries, timeouts, concurrency) is your backend's responsibility.

#### Where everything runs

The **agent loop and the sandbox run on the server** (Anthropic-hosted). The only thing
*you* run is the data path: `pymongo` executes **here in this notebook**, as a custom-tool
round-trip. It is exactly the Path A shape from Section 1, just with five tools instead of
one. This notebook process can run on your laptop for development; it only needs outbound
HTTPS to the Claude API and to MongoDB Atlas.

### AP2 mandate setup

Before the agent runs, the **Trusted Surface** generates a signing keypair and attaches
signed AP2 **Checkout Mandate** and **Payment Mandate** JWTs to each pending transaction in
MongoDB. In production the Shopping Agent would have obtained these from the user ahead of
time; here the notebook plays both roles to keep the demo self-contained.

The Trusted Surface's private key stays in Python local state. It never enters the agent
context, a tool result, or the database. The agent only ever sees the *result* of
`verify_mandates`, not the key material. The mandate receipts are stored in a
`mandate_receipts` collection in the same cluster, which makes double-spend detection a
standard `$lookup` rather than a separate system.

In [17]:
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.primitives.asymmetric import ec

# Trusted Surface keypair — ES256 / P-256, generated fresh each run.
ts_private_key = ec.generate_private_key(ec.SECP256R1())
ts_public_key = ts_private_key.public_key()
# AGENT_PK is the Shopping Agent's public identifier (uncompressed X9.62 hex).
# In production the Shopping Agent would have its own keypair; here the notebook
# plays both roles to keep the demo self-contained.
AGENT_PK = ts_public_key.public_bytes(
    serialization.Encoding.X962,
    serialization.PublicFormat.UncompressedPoint,
).hex()
print(f"Trusted Surface keypair ready  agent_pk={AGENT_PK[:16]}…")

Trusted Surface keypair ready  agent_pk=046cdcba60dbeeda…


Now attach signed mandates to the five pending cases, one per decision lane.
`txn-review-struct` doubles as the seeded human-override case under `AUTO_APPROVE`.

In [18]:
PENDING = [
    "txn-review-clean",
    "txn-review-fraud",
    "txn-review-struct",  # seeded human-override case under AUTO_APPROVE
    "txn-review-high",
    "txn-review-ring",
]
mandates = attach_mandates(coll, PENDING, agent_pk=AGENT_PK, ts_private_key=ts_private_key)
print(f"attached AP2 mandates to {len(mandates)} transactions")

attached AP2 mandates to 5 transactions


### The custom tools (host-side)

Each tool the agent can call is a `type: "custom"` tool. When the agent calls one, the
session pauses, **this notebook** runs the corresponding handler function with `pymongo`, and
sends the JSON result back. The agent never touches the database or the connection string.

- `verify_mandates`: validate the AP2 Checkout and Payment Mandate JWTs (signature,
  constraints, double-spend) before behavioral analysis. This is the hard gate that runs
  first.
- `get_transaction`: fetch the record under review.
- `hybrid_search_similar_frauds`: vector plus full-text RRF over decided history (the case
  under review is filtered out), with the optional reranker. Pattern 3 as a tool.
- `detect_fraud_ring`: `$graphLookup` ring/mule signal. Pattern 4 as a tool, a *separate*
  signal that is not fused into RRF.
- `record_decision`: persist an immutable decision and audit event, then advance the status.
- `escalate`: hand the case to a human (the gate); handled by the notebook, not a DB call.

In [19]:
TOOLS = [
    {
        "type": "custom",
        "name": "verify_mandates",
        "description": "Validate the AP2 Checkout and Payment Mandate JWTs for the transaction — "
        "signature integrity, constraint satisfaction (amount ≤ limit, correct category, within "
        "time window), and double-spend detection. Run this FIRST, before hybrid_search_similar_frauds. "
        "If valid=false, constraints_satisfied=false, or double_spend_detected=true, reject immediately.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "get_transaction",
        "description": "Fetch the full transaction record under review.",
        "input_schema": {
            "type": "object",
            "properties": {"transaction_id": {"type": "string"}},
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "hybrid_search_similar_frauds",
        "description": "Retrieve the most similar prior (already-decided) cases as precedent, "
        "using hybrid vector + full-text search.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "k": {
                    "type": "integer",
                    "description": "how many precedents to return (default 5)",
                },
            },
            "required": ["transaction_id"],
        },
    },
    {
        "type": "custom",
        "name": "detect_fraud_ring",
        "description": "Trace the account's sender->recipient chain for circular-flow / "
        "money-mule / layering patterns.",
        "input_schema": {
            "type": "object",
            "properties": {"account_id": {"type": "string"}},
            "required": ["account_id"],
        },
    },
    {
        "type": "custom",
        "name": "record_decision",
        "description": "Persist the final approve/reject decision with reasoning and an audit event.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "risk_factors": {"type": "array", "items": {"type": "string"}},
                "reasoning": {"type": "string"},
                "escalated": {"type": "boolean"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
            },
            "required": ["transaction_id", "decision", "reasoning"],
        },
    },
    {
        "type": "custom",
        "name": "escalate",
        "description": "Send a risky case to a human reviewer for the final approve/reject "
        "decision. Use for medium-confidence, high-value, structuring, or fraud-ring cases.",
        "input_schema": {
            "type": "object",
            "properties": {
                "transaction_id": {"type": "string"},
                "recommended_decision": {"type": "string", "enum": ["approve", "reject"]},
                "confidence": {"type": "number"},
                "reason": {"type": "string"},
            },
            "required": ["transaction_id", "recommended_decision", "reason"],
        },
    },
]

The host-side handlers map each data tool to its implementation. The reranker second stage
is wired in only when `ENABLE_RERANK` is set and a provider key was found, and on an approval
`_record` also stores the AP2 mandate receipt that later powers double-spend detection.

In [20]:
# Host-side handlers — map each data tool to its handler function.
reranker = (
    (lambda q, ds, top_k: rerank(q, ds, client=ai_client, top_k=top_k))
    if (ENABLE_RERANK and ai_client)
    else None
)


def _verify_mandates(inp):
    result = tool_verify_mandates(db, inp["transaction_id"], ts_public_key)
    status = (
        "pass"
        if result["valid"]
        and result["constraints_satisfied"]
        and not result["double_spend_detected"]
        else "FAIL"
    )
    print(f"  [verify_mandates] {inp['transaction_id']}: {status}")
    return result


def _hybrid(inp):
    return tool_hybrid_search_similar_frauds(
        coll,
        inp["transaction_id"],
        inp.get("k", 5),
        use_rank_fusion=USE_RANK_FUSION,
        enable_rerank=ENABLE_RERANK,
        reranker=reranker,
    )


def _record(inp):
    result = tool_record_decision(
        db,
        inp["transaction_id"],
        inp["decision"],
        confidence=inp.get("confidence", 0),
        risk_factors=inp.get("risk_factors", []),
        reasoning=inp.get("reasoning", ""),
        reviewed_by="human" if inp.get("escalated") else "agent",
        escalated=inp.get("escalated", False),
        recommended_decision=inp.get("recommended_decision"),
    )
    if inp["decision"] == "approve":
        txn_doc = coll.find_one(
            {"transaction_id": inp["transaction_id"]},
            {"checkout_mandate_jwt": 1, "mandate_id": 1, "agent_pk": 1},
        )
        if txn_doc and txn_doc.get("mandate_id"):
            checkout_hash = hashlib.sha256(txn_doc["checkout_mandate_jwt"].encode()).hexdigest()
            store_mandate_receipt(
                db,
                txn_doc["mandate_id"],
                txn_doc["agent_pk"],
                checkout_hash,
                "approve",
            )
    return result


HANDLERS = {
    "verify_mandates": _verify_mandates,
    "get_transaction": lambda inp: tool_get_transaction(coll, inp["transaction_id"]),
    "hybrid_search_similar_frauds": _hybrid,
    "detect_fraud_ring": lambda inp: tool_detect_fraud_ring(coll, inp["account_id"]),
    "record_decision": _record,
}

### Create the agent, environment, and session

`model` / `system` / `tools` live on the **agent** (created once). The session references
it and provisions the sandbox. Networking is `limited`: the agent reaches MongoDB Atlas only
through the host-side round-trip, not directly.

In [21]:
SYSTEM = """You are a financial fraud reviewer. You will be given transaction IDs to review.

For EACH transaction, in order:
1. Call get_transaction to read it.
2. Call verify_mandates to validate the AP2 mandate chain (signature, constraints, double-spend).
   HARD GATE: if verify_mandates returns valid=false, constraints_satisfied=false, OR
   double_spend_detected=true, call record_decision immediately with decision="reject" and
   skip the remaining steps for that transaction.
3. Call hybrid_search_similar_frauds to retrieve similar decided precedents.
4. Call detect_fraud_ring on the sender's account_number to check for ring/mule patterns.
5. Weigh the precedents, the ring signal, the amount, and your confidence. Then make EXACTLY
   ONE terminal call for that transaction:
   - You MUST call escalate (do NOT call record_decision yourself) whenever ANY of these holds:
       * a structuring amount ($4,900-$4,999),
       * a high-value amount (>= $50,000) that you would otherwise approve,
       * detect_fraud_ring reports suspicious_patterns, or
       * your confidence is medium (~75-85).
     Give your recommended_decision and a short reason.
   - Otherwise (a clear-cut case matching none of the above), call record_decision
     (approve/reject).

When you escalate, you will receive the human's decision. Then call record_decision with that
decision, escalated=true, and recommended_decision set to what you had recommended.

Be concise. Move to the next transaction after recording a decision."""

agent = client.beta.agents.create(
    name="MongoDB Atlas fraud reviewer",
    model=MODEL,
    system=SYSTEM,
    tools=TOOLS,
)

environment = client.beta.environments.create(
    name="fraud-review-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

session = client.beta.sessions.create(
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    environment_id=environment.id,
    title="Fraud review",
)
print("session:", session.id)
print(f"Watch in Console: https://platform.claude.com/workspaces/default/sessions/{session.id}")

session: sesn_014BdeLKVmtQBbkj5xBB5vYJ
Watch in Console: https://platform.claude.com/workspaces/default/sessions/sesn_014BdeLKVmtQBbkj5xBB5vYJ


### Run the review with the human gate

We open the event stream, send the queue of pending transactions, and drive the gate loop
with `run_gate_loop`: data-tool calls are serviced automatically from `HANDLERS`, and an
`escalate` call pauses for a human decision.

When `AUTO_APPROVE` is set (e.g. CI), the gate is resolved deterministically, and it
**overrides the agent on the seeded structuring case** so a human verdict that differs from
the agent's is visible in the output. Otherwise you are prompted inline.

In [22]:
OVERRIDE_IDS = {"txn-review-struct"}


def resolver(tool_input):
    # Return the human verdict for an escalated case.
    txn_id = tool_input.get("transaction_id", "")
    recommended = tool_input.get("recommended_decision", "reject")
    reason = tool_input.get("reason", "")
    if AUTO_APPROVE:
        decision = resolve_human_decision(
            recommended, auto_approve=True, override_ids=OVERRIDE_IDS, txn_id=txn_id
        )
        flag = "  <-- HUMAN OVERRIDE" if decision != recommended else ""
        print(f"  [human/auto] {txn_id}: agent recommended {recommended} -> human {decision}{flag}")
        return decision
    print(f"\n  ESCALATED {txn_id} (agent recommends {recommended}): {reason}")
    answer = input("  Your decision [approve/reject]: ").strip().lower()
    return "approve" if answer.startswith("a") else "reject"


def send_result(custom_tool_use_id, result):
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.custom_tool_result",
                "custom_tool_use_id": custom_tool_use_id,
                "content": [{"type": "text", "text": json.dumps(result)}],
            }
        ],
    )


kickoff = {
    "type": "user.message",
    "content": [
        {"type": "text", "text": "Review these flagged transactions: " + ", ".join(PENDING)}
    ],
}

run_start = datetime.now(UTC)

with client.beta.sessions.events.stream(session_id=session.id) as stream:
    client.beta.sessions.events.send(session_id=session.id, events=[kickoff])
    outcome = run_gate_loop(stream, send_result, handlers=HANDLERS, resolver=resolver)

wait_for_idle_status(client, session.id)
print(f"\nserviced {len(outcome['serviced'])} tool calls")

  [verify_mandates] txn-review-clean: pass
  [verify_mandates] txn-review-fraud: pass
  [verify_mandates] txn-review-struct: pass

  ESCALATED txn-review-struct (agent recommends reject): Structuring pattern: $4950 cash deposit (within $4900-$4999 range), third identical deposit this week from same account. Designed to evade CTR reporting threshold.
  [verify_mandates] txn-review-high: pass

  ESCALATED txn-review-high (agent recommends approve): High-value wire ($75,000) from business supplier Northwind to Pacific Logistics, documented bulk shipment. No fraud signals, no ring patterns. Matches approved high-value precedent (txn-high-01). Requires manual approval due to amount.
  [verify_mandates] txn-review-ring: pass

  ESCALATED txn-review-ring (agent recommends reject): Fraud ring detected: $880 transfer continues circular flow pattern among Quartz Trading → Onyx Imports → Slate Ventures accounts. Network of 4 accounts, confirmed circular flow and suspicious patterns. Matches rejec

### Review the outcomes

Read the decisions and audit trail back from MongoDB. The decision and audit collections
are **append-only** (this run never deletes history), so we scope the read-back to records
written during this run. We expect a mix across lanes, and under `AUTO_APPROVE`, at least
one `escalated_to_human` event whose `human_decision` differs from the agent's
recommendation (the visible override). The closing asserts make this a **self-check**: a
re-run that doesn't reproduce the expected outcome fails loudly instead of shipping a
weaker demo.

In [23]:
from collections import Counter

# Append-only collections: scope the read-back to records written during this run.
decisions = list(db["transaction_decisions"].find({"created_at": {"$gte": run_start}}, {"_id": 0}))
audits = list(db["audit_events"].find({"timestamp": {"$gte": run_start}}, {"_id": 0}))

print("Decisions by lane:")
lanes = Counter()
for d in decisions:
    txn = coll.find_one({"transaction_id": d["transaction_id"]})
    lane = txn["lane"] if txn else "?"
    lanes[f"{lane} -> {d['decision']}"] += 1
for k, v in sorted(lanes.items()):
    print(f"  {k}: {v}")

overrides = [
    a
    for a in audits
    if a["event_type"] == "escalated_to_human"
    and a["event_data"].get("human_decision") != a["event_data"].get("recommended_decision")
]
print(
    f"\nescalated_to_human events: {sum(1 for a in audits if a['event_type'] == 'escalated_to_human')}"
)
print(f"human overrides (verdict != agent recommendation): {len(overrides)}")
for a in overrides:
    ed = a["event_data"]
    print(
        f"  {a['transaction_id']}: agent {ed['recommended_decision']} -> human {ed['human_decision']}"
    )

# Self-check — fail loudly if this run didn't produce the teaching outcome, so a future
# re-execution can't silently ship a weaker demo.
decided_lanes = {coll.find_one({"transaction_id": d["transaction_id"]})["lane"] for d in decisions}
assert decided_lanes == {"clean_approve", "clear_reject", "high_value", "ring", "structuring"}, (
    f"expected all 5 lanes decided, saw {sorted(decided_lanes)}"
)
assert len(decisions) == len(PENDING), f"expected {len(PENDING)} decisions, got {len(decisions)}"
assert sum(1 for a in audits if a["event_type"] == "escalated_to_human") >= 1, (
    "expected at least one escalation to a human"
)
if AUTO_APPROVE:
    assert overrides, "AUTO_APPROVE run should show the seeded override on txn-review-struct"
print("\nself-check OK")

Decisions by lane:
  clean_approve -> approve: 1
  clear_reject -> reject: 1
  high_value -> reject: 1
  ring -> reject: 1
  structuring -> approve: 1

escalated_to_human events: 3
human overrides (verdict != agent recommendation): 2
  txn-review-struct: agent reject -> human approve
  txn-review-high: agent approve -> human reject

self-check OK


### MongoDB Atlas as the system of record and audit backbone

Step back from the lane mix and look at where the run's state lives. Everything the agent did
is durable in the same cluster it retrieved from. That is the single-engine payoff on the
write side:

- **Operational state.** `transactions` is the system of record. Each case starts in the
  seeded `pending` status, and `record_decision` advances it to the final `approve` or
  `reject`, so "what is the state of case X?" is a single document read, not a log
  reconstruction.
- **Decisions.** `transaction_decisions` stores one immutable record per verdict: decision,
  confidence, risk factors, reasoning, and who decided (`agent` or `human`).
- **Audit trail.** `audit_events` is **append-only**. Every verdict lands as exactly one
  audit event: `decision_stored` for a direct agent decision, or `escalated_to_human`, which
  captures both what the agent recommended and what the human decided. The override above is
  reconstructable from the trail alone.

Append-only here is enforced at the application level: the decision tools only insert, and
this run never deletes history (which is why the read-back was scoped to `run_start`). For
stronger guarantees, layer on MongoDB Atlas's immutability controls. One case tells the whole story:

In [24]:
# One case, end to end, from the system of record: the operational record, the decision,
# the append-only audit trail, and the AP2 mandate receipt — all in the same cluster.
case = "txn-review-struct"
txn = coll.find_one({"transaction_id": case}, {"_id": 0, "embedding": 0})
print(f"operational record: {case}  lane={txn['lane']}  status={txn['status']}")

decision = db["transaction_decisions"].find_one(
    {"transaction_id": case, "created_at": {"$gte": run_start}}, {"_id": 0}
)
print(
    f"decision record:    {decision['decision']}  reviewed_by={decision['reviewed_by']}  "
    f"confidence={decision['confidence_score']}"
)

print("audit trail (append-only):")
for a in (
    db["audit_events"]
    .find({"transaction_id": case, "timestamp": {"$gte": run_start}}, {"_id": 0})
    .sort("timestamp", 1)
):
    extra = ""
    if a["event_type"] == "escalated_to_human":
        ed = a["event_data"]
        extra = f"  (agent recommended {ed['recommended_decision']}, human decided {ed['human_decision']})"
    print(f"  {a['timestamp']:%H:%M:%S}  {a['event_type']}  severity={a['severity']}{extra}")

# AP2 mandate receipt — written on approval, used for double-spend detection on reuse.
txn_mandate = coll.find_one({"transaction_id": case}, {"mandate_id": 1, "agent_pk": 1})
if txn_mandate and txn_mandate.get("mandate_id"):
    receipt = db["mandate_receipts"].find_one({"mandate_id": txn_mandate["mandate_id"]}, {"_id": 0})
    if receipt:
        print(
            f"mandate receipt:    mandate_id={receipt['mandate_id'][:12]}…  "
            f"agent_pk={receipt['agent_pk'][:12]}…  decision={receipt['decision']}"
        )

operational record: txn-review-struct  lane=structuring  status=approve
decision record:    approve  reviewed_by=human  confidence=0.92
audit trail (append-only):
  21:42:20  escalated_to_human  severity=warning  (agent recommended reject, human decided approve)
mandate receipt:    mandate_id=mandate_0f65…  agent_pk=046cdcba60db…  decision=approve


### Webhooks for production (pointer, not run here)

The streaming loop above is great for development but does not survive a restart. For
production, register a webhook for `session.status_idled`: when the session idles awaiting
a human, your handler queues it for a reviewer and later POSTs the
`user.custom_tool_result`. The **data-tool handlers are identical**; only the trigger
changes. The session is durable server-side, so the human-wait survives a restart; durable
multi-step orchestration (retries, timeouts, concurrency) lives in your own backend. See
[`CMA_operate_in_production.ipynb`](CMA_operate_in_production.ipynb) for the webhook
pattern.

#### What this cookbook keeps simple

It models the core decisioning + human-gate loop. It does not include a request-ingestion
API, a reviewer dashboard, fund-holds / ledger mechanics, or a rules-engine pre-filter. Add
those in your own application around this pattern.

### Cleanup

In [25]:
client.beta.sessions.archive(session.id)
client.beta.environments.archive(environment.id)
# Agents are reusable across runs; archiving is optional and permanent.
# client.beta.agents.archive(agent.id)
mongo.close()
print("archived session + environment; closed MongoDB connection")

archived session + environment; closed MongoDB connection


## Recap

This cookbook took MongoDB Atlas from connection to system of record in one continuous
build, and you can now:

- **Connect** a Claude Managed Agent to MongoDB Atlas three ways (host-side custom tool,
  self-hosted sandbox, self-hosted MCP), with the credential on your side of the boundary
  on every path.
- **Retrieve** with the four patterns that ground an agent (vector, full-text, hybrid
  reciprocal rank fusion, and graph traversal) as liftable pipeline builders.
- **Gate** risky decisions behind CMA's native `requires_action` pause and record the human
  verdict with an audit trail.
- **Persist** decisions, an append-only audit trail, and AP2 mandate receipts to the same
  MongoDB Atlas cluster the agent retrieves from.

## Where this goes next

**AP2, the Agent Payments Protocol,** is the emerging standard for what happens *before* the
fraud reviewer sees a transaction. A Shopping Agent assembles a cart, a Trusted Surface gets
the user to sign a cryptographic **Checkout Mandate** and **Payment Mandate**, and those
signed JWTs are what the Merchant (and its risk layer) must verify. The fraud-review agent
you just ran sits at exactly that boundary: it is the Merchant's risk layer in AP2 terms.

The [AP2 specification](https://github.com/google-agentic-commerce/AP2) (v0.2, 60+ supporting
organizations) defines five roles: **Shopping Agent** (initiates, builds the cart),
**Trusted Surface** (deterministic; signs mandates after getting user consent), **Credential
Provider** (issues the payment token against the Payment Mandate), **Merchant** (verifies the
Checkout Mandate), and **Merchant Payment Processor** (verifies the Payment Mandate). The
fraud-review agent straddles the last two.

**This cookbook already implements that Merchant-side AP2 layer.** Section 3's
`verify_mandates` is the deterministic, no-LLM-reasoning verification tool: JWT signatures,
constraint checks (amount ≤ the user's pre-approved limit, correct category, within the time
window), and double-spend detection against `mandate_receipts`. It runs *before* the
model sees the case. The mandates and receipts live in the same MongoDB Atlas cluster as the
fraud-review trail, so a run produces a **two-layer audit trail in one cluster**: the AP2
mandate chain (authorization, *was the user's Shopping Agent within the granted
constraints?*) alongside the fraud-review trail (risk, *does the transaction have behavioral
red flags?*). Both are queryable with the same pipeline language, backed by the same MongoDB
Atlas indexes, with no new system to operate. The human gate is already AP2's deterministic
final-authorization step for the **Human Not Present** flow: when the human approves, they
are fulfilling the Trusted Surface role, and the `escalated_to_human` audit event is the
receipt.

What remains for full AP2:

1. **Bind the human approval to the mandate.** Add the mandate JWT hash to the
   `escalated_to_human` audit event so the approval is cryptographically tied to the exact
   transaction content the human saw.
2. **Ring detection on `agent_pk`.** Extend the `$graphLookup` signal so that if the same
   `agent_pk` appears in the graph of a known fraud ring, that Shopping Agent's key is
   treated as compromised regardless of this transaction's amount.
3. **The full trust fabric.** The Trusted Agent Provider trust model and the OpenID4VP
   credential issuance flow involve a PKI beyond what a cookbook can stand up inline.

The AP2 spec and reference implementations are at
[github.com/google-agentic-commerce/AP2](https://github.com/google-agentic-commerce/AP2).
